In [1]:
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_FILE = (
    PROJECT_ROOT
    / "data"
    / "exploration"
    / "urgences_horaires.csv"
)

print(f"Source : {SOURCE_FILE.relative_to(PROJECT_ROOT)}")
print(f"Fichier trouvé : {SOURCE_FILE.exists()}")

Source : data\exploration\urgences_horaires.csv
Fichier trouvé : True


## Chargement initial

Cette étape charge le fichier tel qu’il est publié par le MSSS.  
Aucune transformation métier n’est appliquée. L’objectif est de vérifier sa structure réelle avant de définir les règles de nettoyage.

In [3]:
data_raw = pd.read_csv(
    SOURCE_FILE,
    encoding="cp1252",
    encoding_errors="strict",
)

print(f"Nombre de lignes : {data_raw.shape[0]:,}")
print(f"Nombre de colonnes : {data_raw.shape[1]}")

Nombre de lignes : 137
Nombre de colonnes : 17


In [4]:
data_raw[
    [
        "Region",
        "Nom_etablissement",
        "Nom_installation",
    ]
    ].head(10)

,Region,Nom_etablissement,Nom_installation
0,Bas-Saint-Laurent,Total régional,Total régional
1,Bas-Saint-Laurent,SANTÉ QUÉBEC BAS-SAINT-LAURENT,CENTRE HOSPITALIER RÉGIONAL DU GRAND-PORTAGE
2,Bas-Saint-Laurent,SANTÉ QUÉBEC BAS-SAINT-LAURENT,HÔPITAL DE MATANE
3,Bas-Saint-Laurent,SANTÉ QUÉBEC BAS-SAINT-LAURENT,CLSC DE POHÉNÉGAMOOK
4,Bas-Saint-Laurent,SANTÉ QUÉBEC BAS-SAINT-LAURENT,HÔPITAL NOTRE-DAME-DE-FATIMA
5,Bas-Saint-Laurent,SANTÉ QUÉBEC BAS-SAINT-LAURENT,HÔPITAL DE NOTRE-DAME-DU-LAC
6,Bas-Saint-Laurent,SANTÉ QUÉBEC BAS-SAINT-LAURENT,HÔPITAL D'AMQUI
7,Bas-Saint-Laurent,SANTÉ QUÉBEC BAS-SAINT-LAURENT,HÔPITAL RÉGIONAL DE RIMOUSKI
8,Bas-Saint-Laurent,SANTÉ QUÉBEC BAS-SAINT-LAURENT,CENTRE MULTISERVICES DE SANTÉ ET DE SERVICES S...
9,Saguenay-Lac-Saint-Jean,Total régional,Total régional


In [5]:
for position, column in enumerate(data_raw.columns, start=1):
    print(f"{position:02d}. {column!r}")

01. 'RSS'
02. 'Region'
03. 'Nom_etablissement'
04. 'Nom_installation'
05. 'No_permis_installation'
06. 'Nombre_de_civieres_fonctionnelles'
07. 'Nombre_de_civieres_occupees\t\t\t\t      '
08. 'Nombre_de_patients_sur_civiere_plus_de_24_heures'
09. 'Nombre_de_patients_sur_civiere_plus_de_48_heures'
10. 'Nombre_total_de_patients_presents_a_lurgence\t\t\t\t      '
11. 'Nombre_total_de_patients_en_attente_de_PEC'
12. 'DMS_sur_civiere'
13. 'DMS_ambulatoire'
14. 'DMS_sur_civiere_horaire'
15. 'DMS_ambulatoire_horaire\t\t\t\t      '
16. "Heure_de_l'extraction_(image)"
17. ' Mise_a_jour'


### Encodage de la source

Le fichier ne peut pas être décodé strictement en UTF-8.  
L’encodage Windows-1252 (`cp1252`) permet de lire correctement les caractères accentués.

Le pipeline devra déclarer explicitement cet encodage et échouer si le fichier ne peut plus être décodé ainsi. Un changement d’encodage de la source sera considéré comme une modification du contrat technique.

## Normalisation technique

Le jeu de données original est conservé dans `data_raw`.  
Les corrections techniques sont appliquées à une copie nommée `data` afin de préserver une référence fidèle au fichier reçu.

In [6]:
data = data_raw.copy()

In [7]:
data.columns = data.columns.str.strip()

In [8]:
for position, column in enumerate(data.columns, start=1):
    print(f"{position:02d}. {column!r}")

01. 'RSS'
02. 'Region'
03. 'Nom_etablissement'
04. 'Nom_installation'
05. 'No_permis_installation'
06. 'Nombre_de_civieres_fonctionnelles'
07. 'Nombre_de_civieres_occupees'
08. 'Nombre_de_patients_sur_civiere_plus_de_24_heures'
09. 'Nombre_de_patients_sur_civiere_plus_de_48_heures'
10. 'Nombre_total_de_patients_presents_a_lurgence'
11. 'Nombre_total_de_patients_en_attente_de_PEC'
12. 'DMS_sur_civiere'
13. 'DMS_ambulatoire'
14. 'DMS_sur_civiere_horaire'
15. 'DMS_ambulatoire_horaire'
16. "Heure_de_l'extraction_(image)"
17. 'Mise_a_jour'


In [9]:
duplicated_columns = data.columns[data.columns.duplicated()].tolist()

print(f"Colonnes dupliquées : {duplicated_columns}")

Colonnes dupliquées : []


In [10]:
text_columns = data.select_dtypes(include=["object", "string"]).columns

for column in text_columns:
    data[column] = data[column].str.strip()

In [11]:
data[text_columns] = data[text_columns].replace("", pd.NA)

In [12]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 137 entries, 0 to 136
Data columns (total 17 columns):
 #   Column                                            Non-Null Count  Dtype
---  ------                                            --------------  -----
 0   RSS                                               137 non-null    int64
 1   Region                                            137 non-null    str  
 2   Nom_etablissement                                 137 non-null    str  
 3   Nom_installation                                  137 non-null    str  
 4   No_permis_installation                            120 non-null    str  
 5   Nombre_de_civieres_fonctionnelles                 137 non-null    str  
 6   Nombre_de_civieres_occupees                       137 non-null    str  
 7   Nombre_de_patients_sur_civiere_plus_de_24_heures  137 non-null    str  
 8   Nombre_de_patients_sur_civiere_plus_de_48_heures  137 non-null    str  
 9   Nombre_total_de_patients_presents_a_lurgence      137 

In [13]:
column_profile = pd.DataFrame(
    {
        "type_detecte": data.dtypes.astype(str),
        "valeurs_manquantes": data.isna().sum(),
        "valeurs_uniques": data.nunique(dropna=False),
    }
)

column_profile

,type_detecte,valeurs_manquantes,valeurs_uniques
RSS,int64,0,17
Region,str,0,17
Nom_etablissement,str,0,31
Nom_installation,str,0,122
No_permis_installation,str,17,121
Nombre_de_civieres_fonctionnelles,str,0,54
Nombre_de_civieres_occupees,str,0,51
Nombre_de_patients_sur_civiere_plus_de_24_heures,str,0,34
Nombre_de_patients_sur_civiere_plus_de_48_heures,str,0,20
Nombre_total_de_patients_presents_a_lurgence,str,0,75


## Granularité des lignes

Le fichier contient des observations d’installations ainsi que des agrégats régionaux et provinciaux. Ces niveaux doivent être distingués pour éviter les doubles comptages.

In [14]:
data["row_type"] = "unknown"

has_permit_number = data["No_permis_installation"].notna()
installation_name = data["Nom_installation"].astype("string").str.casefold()

data.loc[has_permit_number, "row_type"] = "installation"
data.loc[installation_name.eq("total régional"), "row_type"] = "region"
data.loc[installation_name.eq("ensemble du québec"), "row_type"] = "province"

data["row_type"].value_counts(dropna=False)

row_type
installation    120
region           16
province          1
Name: count, dtype: int64

In [15]:
unknown_rows = data.loc[
    data["row_type"].eq("unknown"),
    [
        "RSS",
        "Region",
        "Nom_etablissement",
        "Nom_installation",
        "No_permis_installation",
    ],
]

unknown_rows

,RSS,Region,Nom_etablissement,Nom_installation,No_permis_installation


In [16]:
installations = data.loc[
    data["row_type"].eq("installation")
].copy()

print(f"Lignes d'installation : {len(installations):,}")
print(
    "Installations distinctes : "
    f"{installations['No_permis_installation'].nunique():,}"
)

Lignes d'installation : 120
Installations distinctes : 120


In [17]:
publication_datetime = pd.to_datetime(
    installations["Mise_a_jour"],
    errors="coerce",
)

extraction_time = pd.to_timedelta(
    installations["Heure_de_l'extraction_(image)"],
    errors="coerce",
)

installations["snapshot_datetime"] = (
    publication_datetime.dt.normalize()
    + extraction_time
)

In [18]:
installations[
    [
        "Mise_a_jour",
        "Heure_de_l'extraction_(image)",
        "snapshot_datetime",
    ]
].head()

,Mise_a_jour,Heure_de_l'extraction_(image),snapshot_datetime
1,2026-10-04T18:45,18:00:00,2026-10-04 18:00:00
2,2026-10-04T18:45,18:00:00,2026-10-04 18:00:00
3,2026-10-04T18:45,18:00:00,2026-10-04 18:00:00
4,2026-10-04T18:45,18:00:00,2026-10-04 18:00:00
5,2026-10-04T18:45,18:00:00,2026-10-04 18:00:00


In [19]:
print(
    "Dates de publication invalides :",
    publication_datetime.isna().sum(),
)

print(
    "Heures d'extraction invalides :",
    extraction_time.isna().sum(),
)

Dates de publication invalides : 0
Heures d'extraction invalides : 0


In [20]:
candidate_key = [
    "No_permis_installation",
    "snapshot_datetime",
]

duplicates = installations.loc[
    installations.duplicated(
        subset=candidate_key,
        keep=False,
    )
].sort_values(candidate_key)

print(f"Lignes impliquées dans un doublon : {len(duplicates):,}")

duplicates[
    candidate_key
    + [
        "Nom_installation",
        "Mise_a_jour",
    ]
].head(20)

Lignes impliquées dans un doublon : 0


,No_permis_installation,snapshot_datetime,Nom_installation,Mise_a_jour


## Analyse des mesures numériques

Les mesures sont initialement interprétées comme du texte. Cette section identifie les valeurs non numériques avant toute conversion afin de ne pas transformer silencieusement une anomalie de source en valeur manquante.

In [21]:
measure_columns = [
    "Nombre_de_civieres_fonctionnelles",
    "Nombre_de_civieres_occupees",
    "Nombre_de_patients_sur_civiere_plus_de_24_heures",
    "Nombre_de_patients_sur_civiere_plus_de_48_heures",
    "Nombre_total_de_patients_presents_a_lurgence",
    "Nombre_total_de_patients_en_attente_de_PEC",
    "DMS_sur_civiere",
    "DMS_ambulatoire",
]

non_numeric_values = {
    column: sorted(
        data.loc[
            pd.to_numeric(data[column], errors="coerce").isna()
            & data[column].notna(),
            column,
        ].unique().tolist()
    )
    for column in measure_columns
}

non_numeric_values

{'Nombre_de_civieres_fonctionnelles': ["pas d'information disponible"],
 'Nombre_de_civieres_occupees': ["pas d'information disponible"],
 'Nombre_de_patients_sur_civiere_plus_de_24_heures': ["pas d'information disponible"],
 'Nombre_de_patients_sur_civiere_plus_de_48_heures': ["pas d'information disponible"],
 'Nombre_total_de_patients_presents_a_lurgence': ["pas d'information disponible"],
 'Nombre_total_de_patients_en_attente_de_PEC': ["pas d'information disponible"],
 'DMS_sur_civiere': ["pas d'information disponible"],
 'DMS_ambulatoire': ["pas d'information disponible"]}

In [22]:
unavailable_rows = data.loc[
    data["Nombre_de_civieres_fonctionnelles"].eq(
        "pas d'information disponible"
    ),
    [
        "RSS",
        "Region",
        "Nom_etablissement",
        "Nom_installation",
        "No_permis_installation",
    ],
]

unavailable_rows

,RSS,Region,Nom_etablissement,Nom_installation,No_permis_installation
30,4,Mauricie et Centre-du-Québec,SANTÉ QUÉBEC MAURICIE-ET-CENTRE-DU-QUÉBEC - UN...,CENTRE MULTISERVICES DE SANTÉ ET DE SERVICES S...,51219160
31,4,Mauricie et Centre-du-Québec,SANTÉ QUÉBEC MAURICIE-ET-CENTRE-DU-QUÉBEC - UN...,CENTRE MULTISERVICES DE SANTÉ ET DE SERVICES S...,51219731
32,4,Mauricie et Centre-du-Québec,SANTÉ QUÉBEC MAURICIE-ET-CENTRE-DU-QUÉBEC - UN...,HÔPITAL DU CENTRE-DE-LA-MAURICIE,51220051
33,4,Mauricie et Centre-du-Québec,SANTÉ QUÉBEC MAURICIE-ET-CENTRE-DU-QUÉBEC - UN...,PAVILLON SAINTE-MARIE,51220168
34,4,Mauricie et Centre-du-Québec,SANTÉ QUÉBEC MAURICIE-ET-CENTRE-DU-QUÉBEC - UN...,HÔTEL-DIEU D'ARTHABASKA,51228039
35,4,Mauricie et Centre-du-Québec,SANTÉ QUÉBEC MAURICIE-ET-CENTRE-DU-QUÉBEC - UN...,HÔPITAL SAINTE-CROIX,51228781
36,4,Mauricie et Centre-du-Québec,SANTÉ QUÉBEC MAURICIE-ET-CENTRE-DU-QUÉBEC - UN...,CENTRE MULTISERVICES DE SANTÉ ET DE SERVICES S...,51233583
48,6,Montréal,SANTÉ QUÉBEC NORD-DE-L'ÎLE-DE-MONTRÉAL - UNIVE...,HÔPITAL DU SACRÉ-COEUR-DE-MONTRÉAL,12475976
57,6,Montréal,SANTÉ QUÉBEC NORD-DE-L'ÎLE-DE-MONTRÉAL - UNIVE...,HÔPITAL FLEURY,51228492
58,6,Montréal,SANTÉ QUÉBEC NORD-DE-L'ÎLE-DE-MONTRÉAL - UNIVE...,HÔPITAL JEAN-TALON,51228500


### Interprétation

La valeur `pas d'information disponible` est une valeur métier publiée par la source, et non une erreur de lecture. Elle doit être conservée dans les données brutes. Dans une copie typée destinée à l'analyse, elle est convertie en valeur manquante tout en restant détectable par les contrôles de qualité.

In [23]:
count_columns = measure_columns[:6]
duration_columns = measure_columns[6:]

data_typed = data.copy()

for column in count_columns:
    data_typed[column] = pd.to_numeric(
        data_typed[column],
        errors="coerce",
    ).astype("Int64")

for column in duration_columns:
    data_typed[column] = pd.to_numeric(
        data_typed[column],
        errors="coerce",
    ).astype("Float64")

data_typed[measure_columns].dtypes.to_frame(name="type_converti")

,type_converti
Nombre_de_civieres_fonctionnelles,Int64
Nombre_de_civieres_occupees,Int64
Nombre_de_patients_sur_civiere_plus_de_24_heures,Int64
Nombre_de_patients_sur_civiere_plus_de_48_heures,Int64
Nombre_total_de_patients_presents_a_lurgence,Int64
Nombre_total_de_patients_en_attente_de_PEC,Int64
DMS_sur_civiere,Float64
DMS_ambulatoire,Float64


## Contrôles de cohérence métier

Ces contrôles vérifient des relations logiques entre les mesures. Une violation ne doit pas être corrigée automatiquement : elle doit être conservée, signalée et examinée.

In [24]:
installation_data = data_typed.loc[
    data_typed["row_type"].eq("installation")
].copy()

business_checks = pd.Series(
    {
        "valeur_negative": installation_data[count_columns]
        .lt(0)
        .any(axis=1)
        .sum(),
        "patients_48h_superieurs_aux_24h": (
            installation_data["Nombre_de_patients_sur_civiere_plus_de_48_heures"]
            > installation_data["Nombre_de_patients_sur_civiere_plus_de_24_heures"]
        ).sum(),
        "patients_24h_superieurs_aux_civieres_occupees": (
            installation_data["Nombre_de_patients_sur_civiere_plus_de_24_heures"]
            > installation_data["Nombre_de_civieres_occupees"]
        ).sum(),
        "patients_en_attente_superieurs_aux_patients_presents": (
            installation_data["Nombre_total_de_patients_en_attente_de_PEC"]
            > installation_data["Nombre_total_de_patients_presents_a_lurgence"]
        ).sum(),
        "occupation_positive_sans_civiere_fonctionnelle": (
            installation_data["Nombre_de_civieres_occupees"].gt(0)
            & installation_data["Nombre_de_civieres_fonctionnelles"].eq(0)
        ).sum(),
    },
    name="nombre_de_violations",
)

business_checks.to_frame()

,nombre_de_violations
valeur_negative,0
patients_48h_superieurs_aux_24h,0
patients_24h_superieurs_aux_civieres_occupees,0
patients_en_attente_superieurs_aux_patients_presents,0
occupation_positive_sans_civiere_fonctionnelle,0


## Taux d'occupation

Le taux d'occupation compare les patients sur civière aux civières fonctionnelles. Une valeur supérieure à 100 % est possible, car les patients peuvent occuper des civières de choc ou de débordement qui ne sont pas incluses dans la capacité fonctionnelle publiée. Une capacité égale à zéro produit volontairement une valeur manquante afin d'éviter une division par zéro.

In [25]:
functional_stretchers = installation_data["Nombre_de_civieres_fonctionnelles"]
occupied_stretchers = installation_data["Nombre_de_civieres_occupees"]

installation_data["occupancy_rate_pct"] = (
    occupied_stretchers
    .div(functional_stretchers)
    .mul(100)
    .where(functional_stretchers.gt(0))
)

occupancy_summary = pd.Series(
    {
        "installations_avec_mesure": installation_data["occupancy_rate_pct"].notna().sum(),
        "installations_sans_mesure": installation_data["occupancy_rate_pct"].isna().sum(),
        "installations_au_dessus_de_100_pct": installation_data["occupancy_rate_pct"].gt(100).sum(),
        "taux_minimum_pct": installation_data["occupancy_rate_pct"].min(),
        "taux_maximum_pct": installation_data["occupancy_rate_pct"].max(),
    },
    name="valeur",
)

occupancy_summary.to_frame()

,valeur
installations_avec_mesure,108.000000
installations_sans_mesure,12.000000
installations_au_dessus_de_100_pct,35.000000
taux_minimum_pct,0.000000
taux_maximum_pct,226.415094


## Conclusion de l'exploration initiale

- La source est un fichier CSV encodé en Windows-1252 et mis à jour chaque heure.
- Le fichier observé contient 120 installations, 16 totaux régionaux et un total provincial.
- La granularité analytique visée est une installation d'urgence à une heure d'extraction donnée.
- La clé `(No_permis_installation, snapshot_datetime)` est unique dans cet échantillon, mais devra être validée sur plusieurs collectes.
- La mention `pas d'information disponible` doit être conservée en RAW puis interprétée comme une mesure manquante dans les couches transformées.
- Les règles de cohérence testées ne présentent aucune violation dans cet échantillon.
- Un taux d'occupation supérieur à 100 % est valide selon la définition du MSSS, puisque les civières de choc et de débordement sont exclues des civières fonctionnelles.

Documentation officielle : [Liste des variables et notes méthodologiques](https://www.donneesquebec.ca/recherche/dataset/d4541afe-9391-44bf-a78f-dae3c9cf1217/resource/968af2e2-8c0f-43d4-9f34-100bd8d295a7/download/listevariables_notesmetho_relevehoraireurgence_20230202.pdf).